# LKIPA Covariance Matrix Temporal Reconstruction
---

Here we reconstruct the covariance matrix from time series data

**Notes**
1. By splitting a pixel into chunks of length $\tau$ seconds, the frequency resolution $\Delta f$ of the TL modes is given by:

$$
    \boxed{
        \Delta f = \frac{1}{\tau}
    }
$$
For a resolution of $\Delta f = 10$ kHz, we need to split the pixel into chunks of $\tau = 100 \mu$ seconds each.  

2. Use ```AdcMode.Direct``` for full 2 GHz sampling and no I/Q downconversion. 

3. From real voltage time series, perform real FFT on each chunk to obtain I/Q quadratures. 

4. Generate covariance matrix for each chunk

5. Average covariance matrices over all chunks

In [ ]:
# IMPORTS
# =======
import os
import sys

import matplotlib.pyplot as plt
%matplotlib widget
import numpy as np
from tqdm import tqdm

import importlib
import LKIPA_library as lib
importlib.reload(lib)

# 1. Import quadrature data from hdf5 file

In [ ]:
folder = r'/media/nanophys-meas/DR_BACKUP/Jai LKIPA Data/2026-07/Time series'   # Choose folder for current month and measurement type

file = sorted(                                                                  # Get the most recent hdf5 file in the folder 
        (f for f in os.listdir(folder) if f.endswith('.hdf5')),
        key=lambda f: os.path.getmtime(os.path.join(folder, f))
    )[-1]   

myrun = file.split('.')[0]                                                      # Get the run name from the file name

file = os.path.join(folder, file)

pump_idx = 0                                                                    # set pump index for desired pump amplitude

pump_freq, pump_amp, df, f_NCO, dc_bias, N_pixels, I_arr, Q_arr, t_arr, fs = lib.retrieve_data(folder, file, myrun, pump_idx)

n_samples = len(t_arr)                                                          # Get number of samples in the time series

# 2. Complex FFT to get I/Q for each pixel